In [1]:
import snntorch as snn
import torch
import torch.nn as nn
from snntorch_network.dataloader import *
from snntorch import surrogate
from pathlib import Path
import numpy as np
from tqdm import tqdm
import brevitas.nn as qnn


In [2]:

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
print(f'Using device {device}')

device = torch.device("cpu")

Using device cuda


In [3]:
class EncoderLIF(nn.Module):
    def __init__(self, in_channels, out_channel, vth, beta, in_weight, **kwargs):
        super(EncoderLIF, self).__init__(**kwargs)
        self.linear1 = qnn.QuantLinear(
            in_channels,
            out_channel,
            bias=False,
            weight_bit_width=8
            )
        self.linear1.weight.data = torch.tensor(in_weight)
        # Input neuron population
        self.leaky1 = snn.Leaky(
            beta=beta,
            threshold=vth,
            learn_threshold=True,
            learn_beta=True,
            reset_mechanism='zero',
            reset_delay=False
            )
        
        self.linear1.to(device)
        self.leaky1.to(device)
        self.time_dim = 2

    def forward(self, data):
        self.leaky1.reset_hidden()
        spk_rec = []
        if self.time_dim is not None:
            dims = list(range(data.dim()))           # Creates a list of dimensions
            dims.pop(self.time_dim)                  # Remove the selected dimension
            dims.insert(0, self.time_dim)            # Insert the selected dimension at the front
            data_permuted = data.permute(dims)  # Permute the tensor
        else:
            data_permuted = data
        for slice in data_permuted:
            x = self.linear1(slice)
            x, _ = self.leaky1(x)
            spk_rec.append(x)
            
        batch_out = torch.stack(spk_rec)  

        return batch_out

In [4]:
DATASET_NAME = 'data_watch_subset_0_40.npz'
DATASET_SUBSET = 'custom'
SUBSET_LIST = [0, 1, 4, 8, 9, 10, 14]
TRAIN_FOLDER_NAME = 'Trained'
NUM_WORKERS = 8
NET_INPUT_DIM = 6
BATCH_SIZE = int(512)

dataset = WisdmDatasetParser(f'{Path.home()}/innuce/snntorch_ExInhibitory/data/{DATASET_NAME}', norm=None, class_sublset=DATASET_SUBSET, subset_list=SUBSET_LIST)

val_set = dataset.get_validation_set(shuffle=False)
train_set = dataset.get_training_set(shuffle=False)
test_set = dataset.get_test_set(shuffle=False)

val_dataset = WisdmDataset(val_set)
train_dataset = WisdmDataset(train_set)
test_dataset = WisdmDataset(test_set)

val_loader  = DataLoader(dataset= val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
train_loader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)


(6,)
(6,)
ytrain shape (55404, 18)
yval shape (18468, 18)
ytest shape (18469, 18)
num classes train dataset: 7 occurrences of each class:[3127 3066 3044 3047 3150 3087 2973]
num classes eval dataset: 7 occurrences of each class:[1035  968 1048  996 1110 1053 1007]
num classes test dataset: 7 occurrences of each class:[1046 1061 1048 1036 1076 1026  982]


In [5]:
name = "inibitory_lif_no_encoder_balanced"
dataset_save_name = f"spike_{name}_dataset.npz"
path = f"{Path.home()}/innuce/snntorch_ExInhibitory/data/{name}.npz"
data = np.load(path,allow_pickle=True)

linear1_w= data['linear1']
print(f"linear1: {linear1_w.shape}")
leaky1_vth= data['leaky1_vth']
leaky1_betas= data['leaky1_betas']
leaky1_betas= leaky1_betas if leaky1_betas >= 0 else np.zeros(leaky1_betas.shape)
print(f"leaky1_betas: {leaky1_betas}")
print(f"leaky1_vth: {leaky1_vth}")

linear1: (200, 6)
leaky1_betas: 0.0
leaky1_vth: 2.195924758911133


In [6]:
encoder_lif = EncoderLIF(NET_INPUT_DIM, 200, leaky1_vth, leaky1_betas, linear1_w).to(device)

In [7]:
encoded_samples = []
labels = []

tqdm_dataloader = tqdm(train_loader)
for _, batch in enumerate(tqdm_dataloader): #eval loop
        input_, label = batch
        input_.to(device)
        # output, encoded = nett(input_)
        encoded = encoder_lif(input_)
        encoded_samples.append(encoded.cpu().detach())
        labels.append(label.cpu().detach())
       # print(f"output: {output.shape}")
        #print(f"label: {label.shape}")
torch.cuda.empty_cache()# Assuming tensor_list is your list of tensors with shape [channels, batch, time]
encoded_samples = torch.cat(encoded_samples, dim=1)
labels = torch.cat(labels, dim=0)
# print("Stacked tensor shape:", encoded_samples.shape)
# print("labels shape:", labels.shape)
encoded_samples = np.transpose(encoded_samples,(1, 2, 0))
train_spike_dataset = [encoded_samples.numpy(), labels.numpy()]
# print("Stacked tensor shape:", encoded_samples.shape)
# print("labels shape:", labels.shape)
del encoded_samples
del labels

  0%|          | 0/42 [00:00<?, ?it/s]/home/franzhd/anaconda3/envs/snntorch_ex/lib/python3.10/site-packages/torch/_tensor.py:1624: UserWarning: Named tensors and all their associated APIs are an experimental feature and subject to change. Please do not use them for anything important until they are released as stable. (Triggered internally at /pytorch/c10/core/TensorImpl.h:1935.)
  return super().rename(names)
100%|██████████| 42/42 [00:02<00:00, 20.59it/s]


In [8]:
from snntorch_network.my_network_without_enc import *
#from snntorch_network.my_network import *
spike_dataset = WisdmDataset(train_spike_dataset)

spike_loader  = DataLoader(dataset= spike_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)


grad = surrogate.fast_sigmoid(5) #use slope for HPO

mod_networ = ExInhbitoryNetwork(NET_INPUT_DIM, 200,500, 7, grad,
                    vth_in=1.0, vth_recurrent=1.0, vth_out=1.0, vth_back=1.0,
                    beta_in=0., beta_recurrent=0., beta_back=0., beta_out=0.,
                    # encoder_dim=int(encoder_dim),
                    # vth_enc_value=vth_enc_value, vth_std=vth_std, beta_std=beta_std,
                    drop_recurrent=0., drop_back=0., drop_out=0.,
                    time_dim=2).to(device)
mod_networ.from_npz(path)
# mod_networ.eval()

/home/franzhd/anaconda3/envs/snntorch_ex/lib/python3.10/site-packages/brevitas/nn/mixin/base.py:77: UserWarning: Keyword arguments are being passed but they not being used.
  warn('Keyword arguments are being passed but they not being used.')


In [9]:
def validate_model(model, data_loader, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        tqdm_dataloader = tqdm(data_loader)
        for _, batch in enumerate(tqdm_dataloader):
            inputs, labels = batch
            inputs = inputs.to(device)
            labels = labels.to(device)
            outputs = model(inputs)
            # For spike trains, sum over the time dimension (assumed as dim=2) to obtain spike counts per channel
            # print(f"outputs: {outputs.shape}")
            # print(f"labels: {labels.shape}")
            _, predicted = outputs.sum(dim=0).max(1)
            # print(f"labels: {labels.numpy().shape}")
            # print(f"batch_pred: {batch_pred.numpy().shape}")
            # print(f"batch_pred: {batch_pred.numpy().shape}")
            # Compute current batch accuracy and update tqdm display
            correct += torch.sum(predicted == labels).cpu().data.item()
            # batch_acc = predicted / labels.size(0) * 100
            total += labels.size(0)
    accuracy = correct / total * 100
    print("Validation Accuracy: {:.2f}%".format(accuracy))
    return accuracy

# Example usage
validate_model(mod_networ, spike_loader, device)

  0%|          | 0/42 [00:00<?, ?it/s]/home/franzhd/anaconda3/envs/snntorch_ex/lib/python3.10/site-packages/snntorch/_neurons/leaky.py:211: UserWarning: Defining your `__torch_function__` as a plain method is deprecated and will be an error in future, please define it as a classmethod. (Triggered internally at /pytorch/torch/csrc/utils/python_arg_parser.cpp:316.)
  self.mem = torch.zeros_like(input_, device=self.mem.device)
100%|██████████| 42/42 [00:15<00:00,  2.75it/s]

Validation Accuracy: 99.05%


99.05089792500232

In [10]:
encoded_samples = []
labels = []

tqdm_dataloader = tqdm(val_loader)
for _, batch in enumerate(tqdm_dataloader): #eval loop
        input_, label = batch
        input_.to(device)
        # output, encoded = nett(input_)
        encoded = encoder_lif(input_)
        encoded_samples.append(encoded.cpu().detach())
        labels.append(label.cpu().detach())
       # print(f"output: {output.shape}")
        #print(f"label: {label.shape}")
torch.cuda.empty_cache()# Assuming tensor_list is your list of tensors with shape [channels, batch, time]
encoded_samples = torch.cat(encoded_samples, dim=1)
labels = torch.cat(labels, dim=0)
# print("Stacked tensor shape:", encoded_samples.shape)
# print("labels shape:", labels.shape)
encoded_samples = np.transpose(encoded_samples,(1, 2, 0))
val_spike_dataset = [encoded_samples.numpy(), labels.numpy()]
# print("Stacked tensor shape:", encoded_samples.shape)
# print("labels shape:", labels.shape)
del encoded_samples
del labels

100%|██████████| 15/15 [00:01<00:00, 13.13it/s]


In [11]:
encoded_samples = []
labels = []

tqdm_dataloader = tqdm(test_loader)
for _, batch in enumerate(tqdm_dataloader): #eval loop
        input_, label = batch
        input_.to(device)
        # output, encoded = nett(input_)
        encoded = encoder_lif(input_)
        encoded_samples.append(encoded.cpu().detach())
        labels.append(label.cpu().detach())
       # print(f"output: {output.shape}")
        #print(f"label: {label.shape}")
torch.cuda.empty_cache()# Assuming tensor_list is your list of tensors with shape [channels, batch, time]
encoded_samples = torch.cat(encoded_samples, dim=1)
labels = torch.cat(labels, dim=0)
# print("Stacked tensor shape:", encoded_samples.shape)
# print("labels shape:", labels.shape)
encoded_samples = np.transpose(encoded_samples,(1, 2, 0))
test_spike_dataset = [encoded_samples.numpy(), labels.numpy()]
# print("Stacked tensor shape:", encoded_samples.shape)
# print("labels shape:", labels.shape)
del encoded_samples
del labels

100%|██████████| 15/15 [00:01<00:00, 11.68it/s]


In [ ]:
from snntorch_network.utils import one_hot_encode
np.savez_compressed(f"../data/{dataset_save_name}",
                    arr0=np.transpose(train_spike_dataset[0], axes=(0,2,1)),
                    arr1=one_hot_encode(train_spike_dataset[1], 7),
                    arr2=np.transpose(val_spike_dataset[0], axes=(0,2,1)),
                    arr3=one_hot_encode(val_spike_dataset[1], 7),
                    arr4=np.transpose(test_spike_dataset[0], axes=(0,2,1)),
                    arr5=one_hot_encode(test_spike_dataset[1], 7))